# Ближайшие соседи и масштаб признаков

Распознаём цифру так: ищем в обучающей части самые похожие картинки и берём их ответ.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_04_mnist_knn/data/digits.csv"
)


def find_digits_csv():
    for p in (
        Path("digits.csv"),
        Path("../digits.csv"),
        Path("../../data/digits.csv"),
        Path("../data/digits.csv"),
        Path("../../../data/digits.csv"),
    ):
        if p.exists():
            return p.resolve()
    return DATA_URL


DIGITS_PATH = find_digits_csv()
df = pd.read_csv(DIGITS_PATH)
PIXELS = [c for c in df.columns if c.startswith('p')]

train_part = df.iloc[:1200].reset_index(drop=True)
test_part = df.iloc[1200:].reset_index(drop=True)
train_pixels = train_part[PIXELS].to_numpy().tolist()
train_labels = train_part['label'].tolist()


## 1. Насколько похожи две картинки

Реализуйте `distance(a, b)`: корень из суммы квадратов разностей по всем 64 числам.

**How:** `zip(a, b)` и `sum(...) ** 0.5`.

**Проверка:** для `[0, 0]` и `[3, 4]` расстояние равно 5.

In [ ]:
def distance(a, b):
    return None  # ваш код


assert distance([1, 2, 3], [1, 2, 3]) == 0
assert abs(distance([0, 0], [3, 4]) - 5) < 1e-9
print(round(distance(train_pixels[0], train_pixels[1]), 2))

## 2. Один ближайший сосед

Возьмите картинку `probe = test_part.loc[0, PIXELS].tolist()`. Пройдите циклом по `train_pixels`, найдите номер ближайшей -> `nn_index`, её ответ -> `nn_label`, расстояние -> `nn_dist`.

**Вопрос:** совпал ли `nn_label` с настоящей цифрой `test_part.loc[0, 'label']`?

In [ ]:
probe = test_part.loc[0, PIXELS].tolist()
nn_index = None
nn_label = None
nn_dist = None
assert nn_index is not None and nn_label is not None and nn_dist is not None
assert 0 <= int(nn_index) < len(train_pixels)
assert int(nn_label) in range(10)
assert float(nn_dist) >= 0
print(nn_index, nn_label, round(float(nn_dist), 2), test_part.loc[0, 'label'])

## 3. Голосование трёх соседей

Отсортируйте номера обучающих картинок по расстоянию до `probe` (`sorted(key=...)`), возьмите первые 3 ответа -> `three_labels`, самый частый среди них -> `vote_label`.

**How:** частый элемент списка — `pd.Series(three_labels).value_counts().idxmax()`.

**Checkpoint:** что делать при ничьей 1–1–1?

In [ ]:
three_labels = []
vote_label = None
assert len(three_labels) == 3
assert vote_label is not None and int(vote_label) in range(10)
print(three_labels, vote_label)

## 4. Один признак может съесть все остальные

Добавьте к таблице столбец `ink_thousands` — суммарную яркость, умноженную на 1000. Посчитайте для двух картинок (строки 0 и 1) вклад этого столбца в квадрат расстояния -> `share_from_ink`.

**Идея:** расстояние измеряется в тех единицах, в которых записаны числа.

In [ ]:
row_a = df.loc[0, PIXELS].tolist()
row_b = df.loc[1, PIXELS].tolist()
ink_a = sum(row_a) * 1000
ink_b = sum(row_b) * 1000
share_from_ink = None  # (ink_a - ink_b)^2 / (полный квадрат расстояния)
assert share_from_ink is not None
assert float(share_from_ink) > 0.9
print(round(float(share_from_ink), 4))

## 5. min-max: привести все столбцы к [0, 1]

Реализуйте `scale_min_max(frame)`: из каждого столбца вычесть его минимум и поделить на размах (максимум − минимум).

**Ловушка:** у трёх пикселей значение всегда одно и то же — размах 0, деление даёт `NaN`. Замените нулевой размах на 1.

**How:** `rng = frame.max() - frame.min()`, затем `rng.replace(0, 1)`.

In [ ]:
def scale_min_max(frame):
    return None  # ваш код


scaled_pixels = scale_min_max(df[PIXELS])
assert scaled_pixels is not None
assert int(scaled_pixels.isna().sum().sum()) == 0
assert float(scaled_pixels.min().min()) >= 0 and float(scaled_pixels.max().max()) <= 1
print(scaled_pixels.iloc[:2, :6].round(2))

## 6. Тот же алгоритм библиотекой

Разбейте таблицу (`test_size=0.25`, `random_state=0`, `stratify=df['label']`), обучите `KNeighborsClassifier(n_neighbors=3)` на пикселях и посчитайте долю верных ответов на проверочной части -> `acc_knn`.

Сравните с baseline из пары 24 (~0.10).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

acc_knn = None
assert acc_knn is not None
assert float(acc_knn) > 0.9
print(round(float(acc_knn), 4))

## 7. Эксперимент: сколько соседей спрашивать

На том же разбиении посчитайте точность при `n_neighbors=1` и `n_neighbors=25` -> `acc_1`, `acc_25`.

Запишите в `K_NOTE`, что происходит с ответом, когда соседей слишком много. **Готового ответа нет.**

In [ ]:
acc_1 = None
acc_25 = None
K_NOTE = ''
assert acc_1 is not None and acc_25 is not None
assert float(acc_1) > float(acc_25)
assert len(K_NOTE) > 50
print(round(float(acc_1), 4), round(float(acc_25), 4), K_NOTE)

## 8. Расширение: свой поиск против библиотеки

Для первых 50 картинок `test_part` предскажите ответ своим кодом (1 сосед) -> `my_preds`, затем то же — `KNeighborsClassifier(n_neighbors=1)` на `train_part`.

Доля совпадений -> `agree_share`.

In [ ]:
my_preds = []
agree_share = None
assert len(my_preds) == 50
assert agree_share is not None and float(agree_share) > 0.95
print(round(float(agree_share), 3))